# Bloodflow videos, in the browser

Notebook versions of the video example scripts in `examples/`. Each section renders one video's frames with matplotlib and encodes the MP4 right here in the browser — JupyterLite ships no ffmpeg binary, so the encoding uses ffmpeg.wasm (vendored in this repo, same path as `demo.ipynb`), through the `export_mp4` helper defined below. The first export downloads ~31 MB of WASM; the browser caches it, and the loaded encoder is reused for the rest of the videos.

Run the cells top to bottom, or jump to one section — each section is self-contained apart from the setup and the exporter. Every section notes how its run is trimmed relative to its script, so it finishes in a few minutes in the browser; the full-scale scripts are listed at the end.

In [ ]:
# Shared setup for every video in this notebook: imports and the
# cluster-colouring helper used by the first three videos.
import io
import re

import numpy as np
import matplotlib.pyplot as plt
from matplotlib import colors as mcolors

from bloodflow import core, network
from bloodflow.core import Params
from bloodflow.constriction import (
    Constriction,
    _step_constriction,
    init_positions_constriction,
)

from pyodide.code import run_js
from pyodide.ffi import to_js


def cluster_colors(pos, p, cutoff):
    """Vectorised clustering; singletons grey, clusters tab colours."""
    n = pos.shape[0]
    dx = pos[:, 0][:, None] - pos[:, 0][None, :]
    dx -= p.L * np.round(dx / p.L)  # minimum image in x
    dy = pos[:, 1][:, None] - pos[:, 1][None, :]
    adj = (dx * dx + dy * dy) < cutoff * cutoff
    parent = list(range(n))

    def find(i):
        while parent[i] != i:
            parent[i] = parent[parent[i]]
            i = parent[i]
        return i

    iu = np.triu_indices(n, k=1)
    for i, j in zip(*iu):
        if adj[i, j]:
            ri, rj = find(int(i)), find(int(j))
            if ri != rj:
                parent[ri] = rj
    labels = np.array([find(i) for i in range(n)])
    _, counts = np.unique(labels, return_counts=True)
    size_of = {lab: int(c) for lab, c in zip(np.unique(labels), counts)}
    # Rank clusters by size so the biggest get the most distinct colours.
    ranked = sorted(
        (lab for lab in np.unique(labels) if size_of[lab] >= 2),
        key=lambda lab: -size_of[lab],
    )
    palette = list(mcolors.TABLEAU_COLORS.values()) + list(
        mcolors.CSS4_COLORS.values()
    )
    col_of = {}
    for k, lab in enumerate(ranked):
        col_of[lab] = mcolors.to_rgba(palette[k % len(palette)])
    grey = (0.72, 0.72, 0.72, 1.0)
    return np.array([col_of.get(lab, grey) for lab in labels])

In [ ]:
# In-browser MP4 exporter, shared by every section below.
# JupyterLite ships no ffmpeg binary, so this uses ffmpeg.wasm
# (FFmpeg compiled to WebAssembly), vendored in this repo under
# vendor/ffmpeg/ and served from the same site as the notebook.
# The single-threaded core is used on purpose: GitHub Pages sends no
# cross-origin-isolation headers, so the multi-threaded core cannot
# load here. The encoder is loaded once and reused for every video.
#
# Two crossings need care (both found the hard way):
# - the ESM wrapper spawns a *module* worker, so the core must be the
#   ESM build (@ffmpeg/core dist/esm), not UMD: a module worker has no
#   importScripts, and importing the UMD glue as a module yields no
#   export, so load() hangs forever with no error.
# - Python objects passed to ffmpeg's JS methods arrive as PyProxies,
#   and postMessage cannot clone a PyProxy into ffmpeg's worker
#   (DataCloneError). Every argument crossing that boundary goes
#   through to_js first, turning bytes into a real JS Uint8Array
#   and lists into real JS Arrays.
_FFMPEG = None  # loaded on first use, reused for every video


async def export_mp4(frames_png, fps, out_name):
    """Encode in-memory PNG frames to an MP4, entirely in the browser.

    frames_png is a list of PNG bytes (as produced by the render cells
    below). The MP4 is saved next to this notebook — it appears in the
    file browser (right-click -> Download) and plays inline below.
    The Python-side frame copies are freed once they live in ffmpeg's
    in-memory filesystem, and the temporary frames are deleted from
    ffmpeg after encoding; the simulation data stays in the notebook.
    """
    global _FFMPEG
    n_frames = len(frames_png)
    if n_frames == 0:
        raise RuntimeError("no frames to encode")
    if _FFMPEG is None:
        # ffmpeg.wasm 0.12.15 (@ffmpeg/ffmpeg) + single-threaded core
        # 0.12.10, vendored in vendor/ffmpeg/. The site root is derived
        # from the kernel worker's own URL: the worker script lives
        # under <site>/extensions/, so everything before that is the
        # site root (/bloodflow/ on GitHub Pages, / on a local
        # JupyterLite at the domain root).
        FFMPEG_ESM = "ffmpeg-esm/index.js"   # @ffmpeg/ffmpeg ESM entry
        FFMPEG_WORKER = "ffmpeg-esm/worker.js"
        CORE_JS = "core/ffmpeg-core.js"      # @ffmpeg/core, ESM build
        CORE_WASM = "core/ffmpeg-core.wasm"
        worker_pathname = run_js("self.location.pathname")
        m = re.match(r"^(.*?)/extensions/", worker_pathname)
        if m is not None:
            site_base = m.group(1) + "/"
        else:
            # Fallback (e.g. a main-thread kernel): the lab/tree page URL.
            m = re.match(r"^(.*)/(lab|tree)(?:/|$)", worker_pathname)
            site_base = (m.group(1) if m else "") + "/"
        vendor_base = f"{site_base}files/vendor/ffmpeg"
        # The import and load() URLs must be fully qualified, not
        # site-relative: dynamic import() inside run_js() resolves
        # relative specifiers against the Pyodide runtime module,
        # which loads from cdn.jsdelivr.net, so a site-relative path
        # would be fetched from the CDN and 404. Resolving the path
        # against the worker's own location first yields an absolute
        # site URL, which import()/fetch() then use as-is.
        vendor_base_url = run_js(
            f"new URL('{vendor_base}', self.location.href).href"
        )
        print(f"loading ffmpeg.wasm from {vendor_base_url} ...")
        ffmpeg_mod = await run_js(f"import('{vendor_base_url}/{FFMPEG_ESM}')")
        _FFMPEG = ffmpeg_mod.FFmpeg.new()
        print("loading ffmpeg.wasm core (vendored in the repo; first run downloads ~31 MB)...")
        await _FFMPEG.load({
            "classWorkerURL": f"{vendor_base_url}/{FFMPEG_WORKER}",
            "coreURL": f"{vendor_base_url}/{CORE_JS}",
            "wasmURL": f"{vendor_base_url}/{CORE_WASM}",
        })
        print("ffmpeg.wasm loaded")
    ffmpeg = _FFMPEG
    for i, png in enumerate(frames_png):
        await ffmpeg.writeFile(f"frame_{i:04d}.png", to_js(png))
    frames_png.clear()  # free the Python-side copies
    print(f"wrote {n_frames} frames to ffmpeg's in-memory filesystem")
    rc = await ffmpeg.exec(to_js([
        "-y", "-framerate", str(fps), "-i", "frame_%04d.png",
        "-c:v", "libx264", "-pix_fmt", "yuv420p", "-crf", "20",
        "-movflags", "+faststart", out_name,
    ]))
    if rc != 0:
        raise RuntimeError(f"ffmpeg exited with code {rc}")
    mp4_bytes = bytes((await ffmpeg.readFile(out_name)).to_py())
    with open(out_name, "wb") as f:
        f.write(mp4_bytes)
    print(f"wrote {out_name} ({len(mp4_bytes)} bytes) — "
          "find it in the file browser (right-click -> Download)")
    for i in range(n_frames):
        await ffmpeg.deleteFile(f"frame_{i:04d}.png")
    await ffmpeg.deleteFile(out_name)
    print(f"deleted {n_frames} temporary frames from ffmpeg's filesystem")
    from IPython.display import Video, display
    try:
        display(Video(out_name, embed=True))
    except Exception as exc:  # inline preview is a bonus; the file is already saved
        print(f"(inline preview unavailable: {exc} — download {out_name} from the file browser)")

## 1. Poiseuille vs plug — the headline video

Side by side, same initial positions: no attraction (left) vs strong attraction (right) in a wide channel (H = 100, a/H = 0.01 — a large vessel, 50 cell diameters wide). Without attraction the near-wall particles crawl while the centre races ahead; with attraction the aggregates move together as a plug.

This is `examples/make_video.py`. The script defaults to 12 s at 25 fps (300 frames); this notebook runs 8 s (200 frames) so it finishes in a few minutes in the browser. Same physics, same seed.

In [ ]:
# Section 1 render — adapted from examples/make_video.py.
# Frames are kept in memory as PNG bytes; the export cell below
# encodes them with export_mp4 (no temp dir, no system ffmpeg).
FPS_1 = 25
SECONDS_1 = 8.0

a, H, L, phi = 1.0, 100.0, 100.0, 0.25
n = int(phi * L * H / (np.pi * a * a))
Umax, dt, kappa = 2.0, 0.02, 2.0
n_steps = 6000
n_frames = int(FPS_1 * SECONDS_1)
stride = max(1, n_steps // n_frames)

base = dict(N=n, a=a, H=H, L=L, eta=1.0, Umax=Umax,
            kappa=kappa, kT=0.0, dt=dt, seed=7)
p_off = Params(eps=0.0, **base)
p_on = Params(eps=0.64, **base)  # Mason ~ 16 at the wall shear rate
print(f"N={n}  H={H} (a/H={a / H})  Mason(on)={p_on.mason_number():.1f}")

rng = np.random.default_rng(7)
pos_off = core.init_positions(p_off, rng)
pos_on = pos_off.copy()
rng_off = np.random.default_rng(7)
rng_on = np.random.default_rng(7)

fig, axes = plt.subplots(1, 2, figsize=(12, 5.4), dpi=100)
fig.subplots_adjust(left=0.04, right=0.98, top=0.86, bottom=0.10,
                    wspace=0.08)
titles = ["No attraction (eps = 0)", "Strong attraction (eps = 0.64)"]
scats = []
for ax, title in zip(axes, titles):
    ax.set_xlim(0, L)
    ax.set_ylim(0, H)
    ax.set_aspect("equal")
    ax.set_title(title, fontsize=13)
    ax.set_xlabel("x (flow direction)")
    ax.tick_params(labelsize=9)
axes[0].set_ylabel("y (channel width)")
axes[1].set_yticklabels([])
time_text = fig.suptitle("", fontsize=13)
fig.canvas.draw()
# Marker area (points^2) matching a disk of radius a in data units.
p0 = axes[0].transData.transform((0.0, 0.0))
p1 = axes[0].transData.transform((2.0 * a, 0.0))
r_px = abs(p1[0] - p0[0]) / 2.0
size = (2.0 * r_px * 72.0 / fig.dpi) ** 2
for ax in axes:
    scats.append(ax.scatter([], [], s=size, edgecolors="none"))

cutoff = 2.0 * a + 0.5 / kappa
frames_plug = []
frame = 0
for step in range(1, n_steps + 1):
    core._step(pos_off, p_off, rng_off, 0.0)
    core._step(pos_on, p_on, rng_on, 0.0)
    if step % stride != 0 and step != n_steps:
        continue
    scats[0].set_offsets(pos_off)
    scats[0].set_color(cluster_colors(pos_off, p_off, cutoff))
    scats[1].set_offsets(pos_on)
    scats[1].set_color(cluster_colors(pos_on, p_on, cutoff))
    time_text.set_text(
        f"Poiseuille flow, t = {step * dt:.1f}  |  grey = single particle,"
        " colour = in an aggregate"
    )
    buf = io.BytesIO()
    fig.savefig(buf, format="png")
    frames_plug.append(buf.getvalue())
    frame += 1
plt.close(fig)
print(f"rendered {frame} frames ({len(frames_plug) / FPS_1:.1f} s at {FPS_1} fps)")

In [ ]:
await export_mp4(frames_plug, FPS_1, "bloodflow-poiseuille-vs-plug.mp4")

## 2. Through a constriction

Same side-by-side setup, now through a constricted channel: the throat is 10 wide (5 particle diameters) between x = 40 and x = 60, with cosine ramps back to the full 50-wide channel. Watch aggregates approach the throat, squeeze through, and reform downstream.

This is `examples/make_constriction_video.py`. The script defaults to 12 s with 3000 equilibration steps; this notebook runs 8 s (200 frames) after 2000 equilibration steps. (The script also has a `--init grid` mode that opens on a uniform lattice; the notebook uses the random start.)

In [ ]:
# Section 2 render — adapted from examples/make_constriction_video.py.
FPS_2 = 25
SECONDS_2 = 8.0
EQUIL_STEPS_2 = 2000

a, H_main, H_throat, L, phi = 1.0, 50.0, 10.0, 100.0, 0.25
constr = Constriction(H_main=H_main, H_throat=H_throat, L=L,
                      xc=50.0, core_half=10.0, ramp=15.0)
n = int(phi * L * H_main / (np.pi * a * a))
Umax, dt, kappa = 2.0, 0.02, 2.0
n_steps = 6000
n_frames = int(FPS_2 * SECONDS_2)
stride = max(1, n_steps // n_frames)

base = dict(N=n, a=a, H=H_main, L=L, eta=1.0, Umax=Umax,
            kappa=kappa, kT=0.0, dt=dt, seed=7)
p_off = Params(eps=0.0, **base)
p_on = Params(eps=1.28, **base)
rng = np.random.default_rng(7)
pos_off = init_positions_constriction(p_off, constr, rng)
pos_on = pos_off.copy()
print(f"N={n}  throat={H_throat} (5 diameters)  "
      f"Mason main={p_on.mason_number():.1f}")
rng_off = np.random.default_rng(7)
rng_on = np.random.default_rng(7)
# Equilibration, as in the analysis runs.
for _ in range(EQUIL_STEPS_2):
    _step_constriction(pos_off, p_off, constr, rng_off)
    _step_constriction(pos_on, p_on, constr, rng_on)

xs = np.linspace(0, L, 400)
y_lo, y_hi = constr.y_lo(xs), constr.y_hi(xs)
fig, axes = plt.subplots(1, 2,
                         figsize=(12.0 * L / 100.0, 5.4), dpi=100)
fig.subplots_adjust(left=0.04, right=0.98, top=0.86, bottom=0.10,
                    wspace=0.08)
titles = ["No attraction (eps = 0)", "Strong attraction (eps = 1.28)"]
scats = []
for ax, title in zip(axes, titles):
    ax.fill_between(xs, y_lo, y_hi, color="#dbe7f3", zorder=0)
    ax.plot(xs, y_lo, color="#4a7bb5")
    ax.plot(xs, y_hi, color="#4a7bb5")
    ax.set_xlim(0, L)
    ax.set_ylim(0, H_main)
    ax.set_aspect("equal")
    ax.set_title(title, fontsize=13)
    ax.set_xlabel("x (flow direction)")
    ax.tick_params(labelsize=9)
axes[0].set_ylabel("y (channel width)")
axes[1].set_yticklabels([])
time_text = fig.suptitle("", fontsize=13)
fig.canvas.draw()
p0 = axes[0].transData.transform((0.0, 0.0))
p1 = axes[0].transData.transform((2.0 * a, 0.0))
r_px = abs(p1[0] - p0[0]) / 2.0
size = (2.0 * r_px * 72.0 / fig.dpi) ** 2
for ax in axes:
    scats.append(ax.scatter([], [], s=size, edgecolors="none", zorder=3))

cutoff = 2.0 * a + 0.5 / kappa
frames_constriction = []
frame = 0
for step in range(1, n_steps + 1):
    _step_constriction(pos_off, p_off, constr, rng_off)
    _step_constriction(pos_on, p_on, constr, rng_on)
    if step % stride != 0 and step != n_steps:
        continue
    scats[0].set_offsets(pos_off)
    scats[0].set_color(cluster_colors(pos_off, p_off, cutoff))
    scats[1].set_offsets(pos_on)
    scats[1].set_color(cluster_colors(pos_on, p_on, cutoff))
    time_text.set_text(
        f"Constricted channel, t = {step * dt:.1f}  |  grey = single,"
        " colour = in an aggregate  |  throat between x = 40 and 60"
    )
    buf = io.BytesIO()
    fig.savefig(buf, format="png")
    frames_constriction.append(buf.getvalue())
    frame += 1
plt.close(fig)
print(f"rendered {frame} frames ({len(frames_constriction) / FPS_2:.1f} s at {FPS_2} fps)")

In [ ]:
await export_mp4(frames_constriction, FPS_2, "bloodflow-constriction.mp4")

## 3. Pulsatile flow — the plug breathes but survives

Same side-by-side setup as section 1, but the driving flow beats: Umax(t) = Umean * (1 + amp*sin(2*pi*t/T)). The bottom strip shows the driving waveform over one beat with a marker at the current phase and the instantaneous Mason number, so aggregate breakup and reformation can be read against the heartbeat.

The physics question: with mean Ma = 16, systole peak shear (1.8x mean) drops Ma to ~9 and diastole trough (0.2x mean) raises it to ~80, so the prediction is that the plug breathes but survives. The phase summary printed after the export tests that.

This is `examples/make_pulsatile_video.py`. The script defaults to 12 beats / 24 s (600 frames); this notebook runs 4 beats / 8 s (200 frames).

In [ ]:
# Section 3 render — adapted from examples/make_pulsatile_video.py.
# cluster_fraction is the script's helper: fraction of particles in
# a cluster of size >= 2 (used for the phase summary, not the colours).
def cluster_fraction(pos, p, cutoff):
    """Fraction of particles in a cluster of size >= 2."""
    n = pos.shape[0]
    dx = pos[:, 0][:, None] - pos[:, 0][None, :]
    dx -= p.L * np.round(dx / p.L)
    dy = pos[:, 1][:, None] - pos[:, 1][None, :]
    adj = (dx * dx + dy * dy) < cutoff * cutoff
    np.fill_diagonal(adj, False)
    return float(adj.any(axis=1).mean())


FPS_3 = 25
SECONDS_3 = 8.0
BEATS_3 = 4.0
AMP_3 = 0.8
PERIOD_3 = 100.0

a, H, L, phi = 1.0, 100.0, 100.0, 0.25
n = int(phi * L * H / (np.pi * a * a))
Umean, dt, kappa = 2.0, 0.02, 2.0
n_steps = int(BEATS_3 * PERIOD_3 / dt)
n_frames = int(FPS_3 * SECONDS_3)
stride = max(1, n_steps // n_frames)

base = dict(N=n, a=a, H=H, L=L, eta=1.0, Umax=Umean,
            kappa=kappa, kT=0.0, dt=dt, seed=7,
            pulse_amp=AMP_3, pulse_period=PERIOD_3)
p_off = Params(eps=0.0, **base)
p_on = Params(eps=0.64, **base)
print(f"N={n}  H={H}  mean Mason(on)={p_on.mason_number():.1f}  "
      f"amp={AMP_3}  period={PERIOD_3}  steps={n_steps}  "
      f"frames={n_frames} (stride {stride})")

rng = np.random.default_rng(7)
pos_off = core.init_positions(p_off, rng)
pos_on = pos_off.copy()
rng_off = np.random.default_rng(7)
rng_on = np.random.default_rng(7)

fig = plt.figure(figsize=(12, 6.8), dpi=100)
gs = fig.add_gridspec(2, 2, height_ratios=[5.2, 1.2], hspace=0.45,
                      wspace=0.08, left=0.04, right=0.98,
                      top=0.88, bottom=0.08)
ax0 = fig.add_subplot(gs[0, 0])
ax1 = fig.add_subplot(gs[0, 1])
axw = fig.add_subplot(gs[1, :])
titles = ["No attraction (eps = 0)", "Strong attraction (eps = 0.64)"]
scats = []
for ax, title in zip((ax0, ax1), titles):
    ax.set_xlim(0, L)
    ax.set_ylim(0, H)
    ax.set_aspect("equal")
    ax.set_title(title, fontsize=13)
    ax.set_xlabel("x (flow direction)")
    ax.tick_params(labelsize=9)
ax0.set_ylabel("y (channel width)")
ax1.set_yticklabels([])
time_text = fig.suptitle("", fontsize=13)
p0 = ax0.transData.transform((0.0, 0.0))
p1 = ax0.transData.transform((2.0 * a, 0.0))
r_px = abs(p1[0] - p0[0]) / 2.0
size = (2.0 * r_px * 72.0 / fig.dpi) ** 2
for ax in (ax0, ax1):
    scats.append(ax.scatter([], [], s=size, edgecolors="none"))

# Waveform strip: one beat, static curve + moving phase marker.
phase = np.linspace(0, PERIOD_3, 400)
wave = Umean * (1 + AMP_3 * np.sin(2 * np.pi * phase / PERIOD_3))
axw.plot(phase, wave, color="black", lw=1.5)
axw.axhline(Umean, color="grey", lw=0.8, ls="--")
axw.set_xlim(0, PERIOD_3)
axw.set_ylim(Umean * (1 - AMP_3) * 0.8, Umean * (1 + AMP_3) * 1.15)
axw.set_xlabel("phase within beat (time units)")
axw.set_ylabel("Umax")
axw.tick_params(labelsize=9)
marker, = axw.plot([], [], "ro", ms=6)
wave_text = axw.set_title("", fontsize=11, loc="left")

cutoff = 2.0 * a + 0.5 / kappa
fig.canvas.draw()


def render_pulsatile(step):
    """Update all artists for simulation step `step`. Returns
    (time, phase_fraction, cluster_fraction_on)."""
    t = step * dt
    ph = (t % PERIOD_3) / PERIOD_3
    scats[0].set_offsets(pos_off)
    scats[0].set_color(cluster_colors(pos_off, p_off, cutoff))
    scats[1].set_offsets(pos_on)
    scats[1].set_color(cluster_colors(pos_on, p_on, cutoff))
    beat = int(t // PERIOD_3) + 1
    time_text.set_text(
        f"Pulsatile flow, t = {t:.1f}  (beat {beat})  |  "
        "grey = single particle, colour = in an aggregate"
    )
    u_now = core.pulsatile_umax(p_on, t)
    gamma_now = 4.0 * u_now / H
    ma_now = (p_on.eps * p_on.kappa) / (gamma_now * a * a)
    cos_ph = np.cos(2 * np.pi * ph)
    sin_ph = np.sin(2 * np.pi * ph)
    if cos_ph > 0.05:
        phase_label = "systole, flow rising"
    elif cos_ph < -0.05:
        phase_label = "diastole, flow falling"
    elif sin_ph > 0:
        phase_label = "peak systole"
    else:
        phase_label = "end diastole"
    marker.set_data([(t % PERIOD_3)], [u_now])
    wave_text.set_text(
        f"Driving flow: Umax = {u_now:.2f}, instantaneous Mason = "
        f"{ma_now:.0f} ({phase_label})"
    )
    frac = cluster_fraction(pos_on, p_on, cutoff)
    return t, ph, frac


frames_pulsatile = []
pulsatile_stats = []
frame = 0
for step in range(1, n_steps + 1):
    core._step(pos_off, p_off, rng_off, 0.0, t=(step - 1) * dt)
    core._step(pos_on, p_on, rng_on, 0.0, t=(step - 1) * dt)
    if step % stride != 0 and step != n_steps:
        continue
    t, ph, frac = render_pulsatile(step)
    pulsatile_stats.append((t, ph, frac))
    buf = io.BytesIO()
    fig.savefig(buf, format="png")
    frames_pulsatile.append(buf.getvalue())
    frame += 1
    if frame % 100 == 0:
        print(f"  frame {frame}/{n_frames} (t={t:.0f})")
plt.close(fig)
print(f"rendered {frame} frames ({len(frames_pulsatile) / FPS_3:.1f} s at {FPS_3} fps)")

In [ ]:
await export_mp4(frames_pulsatile, FPS_3, "bloodflow-pulsatile.mp4")

# Phase summary: mean cluster fraction while the flow is rising
# (systole, cos(2*pi*phase) > 0) vs falling (diastole). The full
# 12-beat run gives ~0.92 at systole vs ~0.97 at diastole.
phases = np.array([s[1] for s in pulsatile_stats])
fracs = np.array([s[2] for s in pulsatile_stats])
systole = fracs[np.cos(2 * np.pi * phases) > 0]
diastole = fracs[np.cos(2 * np.pi * phases) < 0]
print(f"mean cluster fraction: systole {systole.mean():.2f} (n={len(systole)}), "
      f"diastole {diastole.mean():.2f} (n={len(diastole)})")

## 4. The attraction ladder — four strengths at once

Four panels stacked vertically, same starting positions, attraction increasing top to bottom: eps = 0.00, 0.32, 0.64, 1.28 (Mason 0, 8, 16, 32 at H = 100, Umax = 2). The channel is 4x longer here (L = 400) and the run is longer, so the age analysis at the end can compare how long particles linger in each case.

Only groups of more than 6 particles are coloured; groups of 6 or fewer stay grey like singletons. Clustering uses a cell list — the O(N^2) loop is far too slow at N = 3183.

This is `examples/stacked_attraction_video.py`. The script defaults to 60,000 steps / 24 s (600 frames); this notebook runs 6,000 steps / 4 s (100 frames) — the ladder still forms, on a shorter clock.

In [ ]:
# Section 4 helpers — copied from examples/stacked_attraction_video.py.
def cluster_colors_fast(pos, p, cutoff, min_size=7):
    """Cell-list clustering; groups of >= min_size coloured, rest grey."""
    n = pos.shape[0]
    cell = cutoff
    nx = max(1, int(p.L // cell))
    ny = max(1, int(p.H // cell))
    ix = (np.floor(pos[:, 0] / cell).astype(int)) % nx
    iy = np.clip(np.floor(pos[:, 1] / cell).astype(int), 0, ny - 1)
    buckets = {}
    for i in range(n):
        buckets.setdefault((int(ix[i]), int(iy[i])), []).append(i)
    parent = list(range(n))

    def find(i):
        while parent[i] != i:
            parent[i] = parent[parent[i]]
            i = parent[i]
        return i

    cut2 = cutoff * cutoff
    for (cx, cy), members in buckets.items():
        for dcx in (-1, 0, 1):
            for dcy in (-1, 0, 1):
                key = ((cx + dcx) % nx, cy + dcy)
                others = buckets.get(key)
                if not others:
                    continue
                for i in members:
                    xi, yi = pos[i, 0], pos[i, 1]
                    for j in others:
                        if j <= i:
                            continue
                        dx = xi - pos[j, 0]
                        dx -= p.L * np.round(dx / p.L)
                        dy = yi - pos[j, 1]
                        if dx * dx + dy * dy < cut2:
                            ri, rj = find(i), find(j)
                            if ri != rj:
                                parent[ri] = rj
    labels = np.array([find(i) for i in range(n)])
    _, counts = np.unique(labels, return_counts=True)
    size_of = {lab: int(c) for lab, c in zip(np.unique(labels), counts)}
    ranked = sorted(
        (lab for lab in np.unique(labels) if size_of[lab] >= min_size),
        key=lambda lab: -size_of[lab],
    )
    palette = list(mcolors.TABLEAU_COLORS.values()) + list(
        mcolors.CSS4_COLORS.values()
    )
    col_of = {}
    for k, lab in enumerate(ranked):
        col_of[lab] = mcolors.to_rgba(palette[k % len(palette)])
    grey = (0.72, 0.72, 0.72, 1.0)
    return np.array([col_of.get(lab, grey) for lab in labels])


def summarize(age):
    mean = float(np.mean(age))
    q25, q75 = np.percentile(age, [25, 75])
    return {
        "mean": mean,
        "std": float(np.std(age)),
        "cv": float(np.std(age) / mean),
        "median": float(np.median(age)),
        "iqr": float(q75 - q25),
        "p95": float(np.percentile(age, 95)),
        "frac_at_max_age": float(np.mean(age >= age.max() - 1e-9)),
        "max": float(age.max()),
    }

In [ ]:
# Section 4 render — adapted from examples/stacked_attraction_video.py.
# No equilibration: frame one is the random uniform start, and the
# age clock runs over the whole simulation.
FPS_4 = 25
SECONDS_4 = 4.0
STEPS_4 = 6000

a, H, L, phi = 1.0, 100.0, 400.0, 0.25
n = int(phi * L * H / (np.pi * a * a))
Umax, dt, kappa = 2.0, 0.02, 2.0
eps_list = [0.0, 0.32, 0.64, 1.28]
n_frames = int(FPS_4 * SECONDS_4)
stride = max(1, STEPS_4 // n_frames)

base = dict(N=n, a=a, H=H, L=L, eta=1.0, Umax=Umax,
            kappa=kappa, kT=0.0, dt=dt, seed=7)
params = [Params(eps=e, **base) for e in eps_list]
names = [f"eps-{e:g}" for e in eps_list]
print(f"N={n}  H={H} L={L}  Mason numbers: "
      f"{[round(p.mason_number(), 1) for p in params]}  "
      f"steps={STEPS_4} ({STEPS_4 * dt:.0f} time units)  "
      f"frames={n_frames} (stride {stride})")

rng = np.random.default_rng(7)
pos0 = core.init_positions(params[0], rng)
positions = [pos0.copy() for _ in params]
rngs = [np.random.default_rng(7) for _ in params]
last_wrap = [np.zeros(n, dtype=int) for _ in params]
n_wraps = [np.zeros(n, dtype=int) for _ in params]

fig, axes = plt.subplots(4, 1, figsize=(12, 12), dpi=80)
fig.subplots_adjust(left=0.04, right=0.98, top=0.94, bottom=0.03,
                    hspace=0.45)
scats = []
for ax, p, name in zip(axes, params, names):
    ax.set_xlim(0, L)
    ax.set_ylim(0, H)
    ax.set_aspect("equal")
    ax.set_title(f"eps = {p.eps:g}  (Mason {p.mason_number():.0f})",
                 fontsize=13)
    ax.tick_params(labelsize=9)
    if ax is not axes[-1]:
        ax.set_xticklabels([])
    else:
        ax.set_xlabel("x (flow direction)")
axes[0].set_ylabel("y (channel width)")
time_text = fig.suptitle("", fontsize=13)
fig.canvas.draw()
p0 = axes[0].transData.transform((0.0, 0.0))
p1 = axes[0].transData.transform((2.0 * a, 0.0))
r_px = abs(p1[0] - p0[0]) / 2.0
size = (2.0 * r_px * 72.0 / fig.dpi) ** 2
for ax in axes:
    scats.append(ax.scatter([], [], s=size, edgecolors="none"))

cutoff = 2.0 * a + 0.5 / kappa
frames_stacked = []
frame = 0
for step in range(1, STEPS_4 + 1):
    for k, p in enumerate(params):
        x_old = positions[k][:, 0].copy()
        dx = core._step(positions[k], p, rngs[k], 0.0)
        wrapped = (x_old + dx >= p.L) | (x_old + dx < 0.0)
        last_wrap[k][wrapped] = step
        n_wraps[k][wrapped] += 1
    if step % stride != 0 and step != STEPS_4:
        continue
    for k, p in enumerate(params):
        scats[k].set_offsets(positions[k])
        scats[k].set_color(cluster_colors_fast(positions[k], p, cutoff))
    time_text.set_text(
        f"Straight channel, t = {step * dt:.0f}  |  grey = single or in"
        " a group of 6 or fewer, colour = in a group of 7+"
    )
    buf = io.BytesIO()
    fig.savefig(buf, format="png")
    frames_stacked.append(buf.getvalue())
    frame += 1
plt.close(fig)
print(f"rendered {frame} frames ({len(frames_stacked) / FPS_4:.1f} s at {FPS_4} fps)")
# Kept for the age analysis below: last wrap step and wrap count
# per particle, per case.
stacked_last_wrap, stacked_n_wraps = last_wrap, n_wraps
stacked_names, stacked_params = names, params
stacked_steps, stacked_dt = STEPS_4, dt

In [ ]:
await export_mp4(frames_stacked, FPS_4, "stacked-attraction-video.mp4")

In [ ]:
# Age analysis at the end of the run: age = time since a particle
# last wrapped around the periodic channel (never-wrapped particles
# get the full run). The script also writes these rows to
# stacked-age.csv; the summaries and histograms below are the same
# numbers.
import csv

stacked_results = {}
for name, p, lw, nw in zip(stacked_names, stacked_params,
                           stacked_last_wrap, stacked_n_wraps):
    age = (stacked_steps - lw) * stacked_dt
    stacked_results[name] = (age, nw)
    print(f"\n[{name}]")
    for k, v in summarize(age).items():
        print(f"  {k}: {v:.4g}")

csv_path = "stacked-age.csv"
with open(csv_path, "w", newline="") as fh:
    w = csv.writer(fh)
    w.writerow(["case", "particle", "age", "n_wraps"])
    for name, (age, nw) in stacked_results.items():
        for i in range(n):
            w.writerow([name, i, f"{age[i]:.4f}", int(nw[i])])
print(f"\nwrote {csv_path}")

fig2, axes2 = plt.subplots(4, 1, figsize=(9, 11), sharex=True)
for ax, name, color in zip(axes2, stacked_names,
                           ["#4a7bb5", "#6a9e6a", "#d95f2b", "#8e44ad"]):
    age = stacked_results[name][0]
    ax.hist(age, bins=25, alpha=0.6, color=color)
    ax.axvline(np.median(age), color=color, linestyle="--")
    ax.set_yscale("log")
    ax.set_title(name)
    ax.set_ylabel("count")
axes2[-1].set_xlabel("particle age (time since last wrap)")
fig2.suptitle("Age at end of run (dashed = median, counts on log scale)")

## 5. The branching network

A symmetric 1→2→4→8→4→2→1 branching ladder with periodic trunk-to-trunk wrap, two panels stacked (no attraction vs strong attraction), same initial positions. The grey background is the channel itself; particles are true size; grey = single, colour = in an aggregate.

This is `examples/make_network_video.py`. The script defaults to 800 time units / 24 s (600 frames) and checkpoints every frame, because the full run takes over an hour — a killed run resumes instead of starting over. This notebook runs 100 time units / 4 s (100 frames) with no checkpoints: short enough to finish in the browser, long enough to see aggregates navigate the branches.

In [ ]:
# Section 5 helper — copied from examples/make_network_video.py.
def cluster_colors_fast_net(pos, cutoff, min_size=2, accent=None):
    """Grid-based clustering.

    Clusters of at least min_size particles are coloured; smaller
    ones stay grey. With accent=None the old rainbow-by-size
    colouring is used (colours are re-ranked every frame, which
    flickers as clusters merge and split); with accent set, every
    qualifying cluster gets that one colour, so nothing switches.
    """
    n = pos.shape[0]
    cell = cutoff
    grid = {}
    for i in range(n):
        grid.setdefault((int(pos[i, 0] // cell), int(pos[i, 1] // cell)),
                        []).append(i)
    parent = list(range(n))

    def find(i):
        while parent[i] != i:
            parent[i] = parent[parent[i]]
            i = parent[i]
        return i

    cut2 = cutoff * cutoff
    for i in range(n):
        cx, cy = int(pos[i, 0] // cell), int(pos[i, 1] // cell)
        for gx in (cx - 1, cx, cx + 1):
            for gy in (cy - 1, cy, cy + 1):
                for j in grid.get((gx, gy), ()):
                    if j <= i:
                        continue
                    dx = pos[j, 0] - pos[i, 0]
                    dy = pos[j, 1] - pos[i, 1]
                    if dx * dx + dy * dy < cut2:
                        ri, rj = find(i), find(j)
                        if ri != rj:
                            parent[ri] = rj
    labels = np.array([find(i) for i in range(n)])
    _, counts = np.unique(labels, return_counts=True)
    size_of = {lab: int(c) for lab, c in zip(np.unique(labels), counts)}
    grey = (0.72, 0.72, 0.72, 1.0)
    if accent is not None:
        acc = mcolors.to_rgba(accent)
        return np.array([acc if size_of[lab] >= min_size else grey
                         for lab in labels])
    ranked = sorted(
        (lab for lab in np.unique(labels) if size_of[lab] >= min_size),
        key=lambda lab: -size_of[lab],
    )
    palette = list(mcolors.TABLEAU_COLORS.values()) + list(
        mcolors.CSS4_COLORS.values()
    )
    col_of = {}
    for k, lab in enumerate(ranked):
        col_of[lab] = mcolors.to_rgba(palette[k % len(palette)])
    return np.array([col_of.get(lab, grey) for lab in labels])

In [ ]:
# Section 5 render — adapted from examples/make_network_video.py.
# No checkpoints and no trajectory dump here: frames are kept in
# memory and encoded by the export cell below.
FPS_5 = 25
SECONDS_5 = 4.0
UNITS_5 = 100.0
DPI_5 = 100

net = network.build_network()
a, kappa, dt = 1.0, 2.0, 0.02
n_steps = int(UNITS_5 / dt)
n_frames = int(FPS_5 * SECONDS_5)
stride = max(1, n_steps // n_frames)

p_tmp = Params(N=1, a=a, H=200.0, L=net.L_total, eps=0.0,
               kappa=kappa, dt=dt, seed=7)
rng = np.random.default_rng(7)
pos0, n, area = network.init_positions_network(net, p_tmp, rng)
print(f"N={n}  area~{area:.0f}  steps={n_steps}  frames={n_frames} "
      f"(stride {stride})")
base = dict(N=n, a=a, H=200.0, L=net.L_total, eta=1.0, Umax=2.0,
            kappa=kappa, kT=0.0, dt=dt, seed=7)
p_off = Params(eps=0.0, **base)
p_on = Params(eps=0.64, **base)
pos_off = pos0.copy()
pos_on = pos0.copy()
rng_off = np.random.default_rng(7)
rng_on = np.random.default_rng(7)

fig, axes = plt.subplots(2, 1, figsize=(14, 7.5), dpi=DPI_5)
fig.subplots_adjust(left=0.04, right=0.98, top=0.90, bottom=0.06,
                    hspace=0.35)
# Channel background: capsules as thick round-capped lines. Points
# per data unit from the axes geometry (no draw needed first).
px_per_unit = 0.94 * 14.0 * DPI_5 / net.L_total
lw_per_width = px_per_unit * 72.0 / 100.0
for ax in axes:
    ax.set_xlim(0, net.L_total)
    ax.set_ylim(0, 200.0)
    ax.set_aspect("equal")
    ax.tick_params(labelsize=9)
    for s in net.segments:
        ax.plot([s.x0, s.x1], [s.y0, s.y1], color="#dddddd",
                lw=s.width * lw_per_width, solid_capstyle="round",
                zorder=0)
axes[0].set_title("No attraction (eps = 0)", fontsize=13)
axes[1].set_title("Strong attraction (eps = 0.64)", fontsize=13)
axes[1].set_xlabel("x (flow direction, periodic trunk-to-trunk wrap)")
axes[0].set_ylabel("y")
axes[1].set_ylabel("y")
time_text = fig.suptitle("", fontsize=13)
size = (2.0 * px_per_unit * 72.0 / 100.0) ** 2
scats = [ax.scatter([], [], s=size, edgecolors="none", zorder=2)
         for ax in axes]
fig.canvas.draw()

cutoff = 2.0 * a + 0.5 / kappa
# Per-particle tracking (level durations and loop times), as in the script.
level_now = [network.dominant_level(pos_off, net),
             network.dominant_level(pos_on, net)]
level_since = [np.zeros(n), np.zeros(n)]
level_dur = [{0: [], 1: [], 2: [], 3: []},
             {0: [], 1: [], 2: [], 3: []}]
loop_start = [np.zeros(n), np.zeros(n)]
loop_times = [[], []]
wraps = [0, 0]


def track(step):
    """Update level durations (every 25 steps) and loop times."""
    t = step * dt
    if step % 25 != 0:
        return
    for k, pos in enumerate((pos_off, pos_on)):
        lv = network.dominant_level(pos, net)
        changed = lv != level_now[k]
        if changed.any():
            for old in range(4):
                m = changed & (level_now[k] == old)
                if m.any():
                    level_dur[k][old].extend(
                        (t - level_since[k][m]).tolist())
            level_now[k][changed] = lv[changed]
            level_since[k][changed] = t


def render_network(step):
    t = step * dt
    for k, pos in enumerate((pos_off, pos_on)):
        scats[k].set_offsets(pos)
        scats[k].set_color(cluster_colors_fast_net(pos, cutoff, 2, None))
    time_text.set_text(
        f"Branching network, t = {t:.1f}  |  loops completed: "
        f"{wraps[0]} (no attraction) / {wraps[1]} (attraction)  |  "
        "grey = single particle, colour = in an aggregate"
    )
    return t


frames_network = []
frame = 0
for step in range(1, n_steps + 1):
    _, w0 = network.step_network(pos_off, p_off, net, rng_off, 0.0)
    _, w1 = network.step_network(pos_on, p_on, net, rng_on, 0.0)
    wraps[0] += int(w0.sum())
    wraps[1] += int(w1.sum())
    for k, w in enumerate((w0, w1)):
        if w.any():
            t = step * dt
            loop_times[k].extend((t - loop_start[k][w]).tolist())
            loop_start[k][w] = t
    track(step)
    if step % stride != 0 and step != n_steps:
        continue
    t = render_network(step)
    buf = io.BytesIO()
    fig.savefig(buf, format="png")
    frames_network.append(buf.getvalue())
    frame += 1
    if frame % 50 == 0:
        print(f"  frame {frame}/{n_frames} (t={t:.0f})")
plt.close(fig)
print(f"rendered {frame} frames ({len(frames_network) / FPS_5:.1f} s at {FPS_5} fps)")
# Kept for the summary in the next cell.
network_wraps, network_loop_times = wraps, loop_times

In [ ]:
await export_mp4(frames_network, FPS_5, "bloodflow-network.mp4")

print(f"loops completed: {network_wraps[0]} (no attraction) / {network_wraps[1]} (attraction)")
for k, name in enumerate(["no attraction", "attraction"]):
    lt = network_loop_times[k]
    if lt:
        print(f"{name}: mean loop time {np.mean(lt):.1f} "
              f"(n={len(lt)}, median {np.median(lt):.1f})")
    else:
        print(f"{name}: no loops completed in this short run")

## Running the full-scale scripts

The sections above are trimmed for the browser: fewer seconds, fewer steps, fewer beats. The physics and the seeds are the same as the scripts; the full versions run on a machine with Python and a system ffmpeg:

```bash
python3 examples/make_video.py                  # section 1, 12 s
python3 examples/make_constriction_video.py     # section 2, 12 s
python3 examples/make_pulsatile_video.py        # section 3, 12 beats
python3 examples/stacked_attraction_video.py    # section 4, 60,000 steps
python3 examples/make_network_video.py          # section 5, 800 time units
```

Each script writes its MP4 next to itself (see `--out`). The notebook versions encode theirs in the browser instead, with the same ffmpeg.wasm path `demo.ipynb` uses — the first export in this notebook downloads ~31 MB of WASM, and the loaded encoder is reused for the remaining videos.

The pre-rendered pulsatile video from the full-scale run is in `videos/bloodflow-pulsatile.mp4`.